# getitem-back-add-at — ex1: getitem_back — index_add_ into zeros_like(x)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `getitem-back-add-at`. Running the final beacon cell reports progress against the `Backprop: getitem_back via add-at` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: getitem_back via add-at` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`getitem-back-add-at`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "getitem-back-add-at"
DD_SUBTOPIC = "Backprop: getitem_back via add-at"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `getitem_back` via add-at — quick refresher

`out = x[idx]` reads a subset of `x`. The backward fn must place `grad_out` into a zero tensor at `idx` — but plain assignment fails when `idx` repeats (you'd overwrite). Use **scatter-add** semantics: any index appearing N times accumulates N contributions.

**Worked exemplar.**
```
x.shape  = (5,)
idx      = [0, 2, 0]            # index 0 appears twice
out      = x[idx]               # out.shape = (3,)
grad_in  = zeros_like(x)
grad_in.index_add_(0, idx, grad_out)   # accumulates at repeated idx
# grad_in == [g[0]+g[2], 0, g[1], 0, 0]
```

Repeat indices = gradients sum. Plain `grad_in[idx] = grad_out` would drop one of the two contributions to position 0.

### Exercise 1 — getitem_back — index_add_ into zeros_like(x)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply scatter-add semantics to place grad_out into a zeros tensor at idx, accumulating contributions for any repeated indices.
> Keywords: getitem, index, scatter-add, index_add
> ```

**KCs targeted:** `getitem-backward-pattern`, `scatter-add-for-repeated-indices`

Implement `getitem_back(grad_out, out, x, idx)` for the forward op `out = x[idx]`. For this drill `idx` is a 1-D `torch.LongTensor` and `x` is 1-D — we're indexing along axis 0.

Derivation:
- `out[i] = x[idx[i]]`, so `d(out[i])/d(x[j]) = 1` if `j == idx[i]` else `0`.
- Chain rule: `dL/dx[j] = sum_i (grad_out[i] if idx[i] == j else 0)`.
- **Repeated indices SUM** — index 0 appearing twice contributes twice.

Implementation:
1. Allocate `grad_in = torch.zeros_like(x)`.
2. Use `grad_in.index_add_(0, idx, grad_out)` to accumulate at the right positions.
3. Return `grad_in`.

**Why not `grad_in[idx] = grad_out`.** With repeated indices, the last write wins — you'd drop one of the contributions. `index_add_` is the scatter-add primitive; it accumulates instead of overwriting.

Return a `torch.Tensor` with the same shape as `x`. No autograd.

In [ ]:
def getitem_back(grad_out: Tensor, out: Tensor, x: Tensor, idx: Tensor) -> Tensor:
    grad_in = t.zeros_like(x)
    # index_add_ accumulates — repeated entries in idx sum into the same row.
    grad_in.index_add_(0, idx, grad_out)
    return grad_in


<details><summary>Solution</summary>

```python
def getitem_back(grad_out: Tensor, out: Tensor, x: Tensor, idx: Tensor) -> Tensor:
    grad_in = t.zeros_like(x)
    # index_add_ accumulates — repeated entries in idx sum into the same row.
    grad_in.index_add_(0, idx, grad_out)
    return grad_in
```

**Repeated-index = summation.** This is the rule that makes scatter-add the right primitive. Whenever multiple output positions read the same input position, their gradient contributions ALL flow back and ADD. Plain assignment drops all but one.

**Why `index_add_` not `scatter_add_`.** Both work for 1-D. `scatter_add_` is more general (supports arbitrary-dim scatter indices), but for the 1-D-along-axis-0 case `index_add_` reads more clearly.

**Conservation.** `g.sum()` must equal `grad_out.sum()` — every unit of gradient that came in must land somewhere in `x`. Useful sanity check when debugging.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()